# Blind final check — marker correlation, cross-batch rank agreement, prediction

Analysis of metric groups **L**, **M** and **N**, computed *after* the harmonization
approaches were selected and deliberately excluded from the Figure 3 clustermap and the
composite score.

| Group | Prefix | Question |
|---|---|---|
| L | `mk_` | Is each marker gene's expression profile across a cohort's samples preserved by harmonization? |
| M | `xb_` | Do same-biology samples from different batches rank the marker genes similarly — and more so than different-biology samples? |
| N | `pv_` | Can a 10-PC logistic regression predict diagnosis in a batch it has never seen, and does the score collapse under permuted labels? |

**Reading guide.** Group L saturates at ~1.0 for any harmonizer that applies a per-batch
monotone transform of each gene, because within-cohort ranks are then unchanged by
construction. A near-1.0 Group L value is therefore *not* evidence of quality; it is a
"nothing broke" guard rail, and the marker-minus-housekeeping contrast is the informative
part. **Group M's margin (`xb_rank_agree_ratio`) carries the discriminative conclusion.**

**Narrow panel variant.** Group L here is read from the `mk_*_narrow_set` columns: the
same aggregates over the 56-gene QC-filtered subset (present in all 42 `01_raw__post0`
matrices, mean `frac_lt_1` < 0.20). The marker-versus-housekeeping margin is reported
twice — against the full housekeeping panel and against `PGK1`, the one housekeeping gene
inside the narrow subset — because only 6 of the 15 housekeeping genes clear the same
coverage bar the narrow markers had to clear. §3b decides which of the two is less noisy.
Every gene-level view is restricted to those 56 genes, taken from the same
`marker_gene_correlations_long` table — the narrow family adds no long table of its own.
Groups M and N are unchanged from the full-panel notebook and are shown here for context.

Generated by `create_correlation_prediction_notebook.py`.

## §1 — Setup and data load

In [ ]:
import warnings
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import mannwhitneyu, pearsonr, median_abs_deviation

warnings.filterwarnings("ignore")

# Publication defaults. 7.5 pt is the floor for every text element in these figures.
FONT_SIZE = 7.5
plt.rcParams.update(
    {
        "font.size": FONT_SIZE,
        "axes.labelsize": FONT_SIZE,
        "axes.titlesize": FONT_SIZE,
        "xtick.labelsize": FONT_SIZE,
        "ytick.labelsize": FONT_SIZE,
        "legend.fontsize": FONT_SIZE,
        "figure.titlesize": FONT_SIZE,
        "pdf.fonttype": "truetype",
        "svg.fonttype": "none",
        "figure.dpi": 200,
        "savefig.bbox": "tight",
    }
)
sns.set_style("ticks")

# Temperature-split palette: cold = normal/reference, warm = tumour/harmonized.
COL_BEST = "#E63946"  # the clustermap winners
COL_REST = "#8D99AE"  # everything else
COL_RAW = "#457B9D"  # unharmonized baseline
COL_PERM = "#CED4DA"  # permuted null band
COL_HK = "#6C757D"  # housekeeping control

# ── Variant ────────────────────────────────────────────────────────────────
NARROW = True  # narrow-panel variant?
MK_SFX = "_narrow_set"  # suffix on every Group L column and figure name

# ── Input paths. Every file this notebook reads is named here and nowhere else. ──
DATE_TAG = "260905"
DATE_TAG_LONG = "260826"
METRIC_TABLES = Path("metric_tables")
METRICS_CSV = METRIC_TABLES / f"metrics_comprehensive_{DATE_TAG}.csv"
GENE_LONG_CSV = METRIC_TABLES / f"marker_gene_correlations_long_{DATE_TAG_LONG}.csv"
COHORT_LONG_CSV = METRIC_TABLES / f"marker_cohort_correlations_long_{DATE_TAG_LONG}.csv"
FOLDS_LONG_CSV = METRIC_TABLES / f"prediction_folds_long_{DATE_TAG_LONG}.csv"
MARKER_ANN_CSV = Path("../harmonization-metrics-calculation/marker_gene_annotation.csv")
COVERAGE_AUDIT_CSV = Path("marker_gene_coverage_audit_260819.csv")
SUPP_FILE3_CSV = Path(
    "../figures_for_article/supplementary_260824/Supplementary File 3.csv.gz"
)

FIG_DIR = Path("../figures_for_article/current_figures_tables_for_article_260819")
FIG_DIR.mkdir(parents=True, exist_ok=True)


def save_figure(fig, name, figures_dir=FIG_DIR):
    """Save one figure as PDF, SVG and PNG, with the variant suffix on the name."""
    name = f"{name}{MK_SFX}"
    figures_dir = Path(figures_dir)
    figures_dir.mkdir(parents=True, exist_ok=True)
    for ext, kw in (("pdf", {}), ("svg",{"transparent": True}), ("png", {"dpi": 200})):
        fig.savefig(figures_dir / f"{name}.{ext}", bbox_inches="tight", **kw)


for _p in [
    METRICS_CSV,
    GENE_LONG_CSV,
    COHORT_LONG_CSV,
    FOLDS_LONG_CSV,
    MARKER_ANN_CSV,
    SUPP_FILE3_CSV,
]:
    print(f"{'OK  ' if _p.exists() else 'MISS'} {_p}")
print("Setup complete.")

### Plotting functions

In [ ]:
FONT_BASE = 7.5
FONT_TITLE = FONT_BASE
FONT_LABEL = FONT_BASE
FONT_TICK = FONT_BASE
FONT_LEGEND = FONT_BASE
FONT_ANNOT = FONT_BASE
FONT_SMALL = FONT_BASE
FONT_TINY = FONT_BASE

plt.rcParams.update(
    {
        "svg.fonttype": "none",
        "figure.dpi": 150,
        "font.size": FONT_BASE,
        "axes.titlesize": FONT_TITLE,
        "axes.labelsize": FONT_LABEL,
        "xtick.labelsize": FONT_TICK,
        "ytick.labelsize": FONT_TICK,
        "legend.fontsize": FONT_LEGEND,
        "figure.titlesize": FONT_TITLE,
    }
)


def scatter_plot(
    data: pd.DataFrame,
    x: str,
    y: str,
    hue: str,
    style: str | None = None,
    size: str | None = None,
    palette: dict | str | None = None,
    hue_order: list | None = None,
    style_order: list | None = None,
    alpha: float = 0.75,
    s: int = 30,
    ax: plt.Axes | None = None,
    figsize: tuple = (8, 6),
    xlabel: str | None = None,
    ylabel: str | None = None,
    title: str | None = None,
    legend_outside: bool = False,
    save_path: str | None = None,
) -> plt.Axes:
    """Scatter plot wrapping sns.scatterplot with standard project styling."""
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    sns.scatterplot(
        data=data,
        x=x,
        y=y,
        hue=hue,
        style=style,
        size=size,
        palette=palette,
        hue_order=hue_order,
        style_order=style_order,
        alpha=alpha,
        s=s,
        linewidth=0,
        ax=ax,
    )
    ax.set_xlabel(xlabel if xlabel is not None else x, fontsize=FONT_LABEL)
    ax.set_ylabel(ylabel if ylabel is not None else y, fontsize=FONT_LABEL)
    if title:
        ax.set_title(title, fontsize=FONT_TITLE)
    ax.tick_params(labelsize=FONT_TICK)
    if legend_outside and ax.get_legend() is not None:
        handles, labels = ax.get_legend_handles_labels()
        ax.get_legend().remove()
        ax.legend(
            handles,
            labels,
            bbox_to_anchor=(1.05, 1),
            loc="upper left",
            fontsize=FONT_LEGEND,
            frameon=False,
        )
    sns.despine(ax=ax)
    if save_path:
        ax.get_figure().savefig(FIGURES_DIR / f"{save_path}.svg", bbox_inches="tight")
        ax.get_figure().savefig(
            FIGURES_DIR / f"{save_path}.png", bbox_inches="tight", dpi=200
        )
    return ax


def bar_plot(
    data: pd.DataFrame,
    x: str,
    y: str,
    hue: str | None = None,
    palette: dict | str | None = None,
    order: list | None = None,
    hue_order: list | None = None,
    estimator: str = "mean",
    errorbar: tuple | None = ("ci", 95),
    errwidth: float = 0.5,
    orient: str = "v",
    ax: plt.Axes | None = None,
    figsize: tuple = (10, 5),
    xlabel: str | None = None,
    ylabel: str | None = None,
    title: str | None = None,
    xtick_rotation: int = 45,
    legend_outside: bool = False,
    save_path: str | None = None,
) -> plt.Axes:
    """Bar plot wrapping sns.barplot with standard project styling."""
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    if orient == "h":
        sns.barplot(
            data=data,
            x=x,
            y=y,
            hue=hue,
            palette=palette,
            order=order,
            hue_order=hue_order,
            estimator=estimator,
            errorbar=errorbar,
            orient="h",
            ax=ax,
            errwidth=errwidth,
        )
    else:
        sns.barplot(
            data=data,
            x=x,
            y=y,
            hue=hue,
            palette=palette,
            order=order,
            hue_order=hue_order,
            estimator=estimator,
            errorbar=errorbar,
            ax=ax,
            errwidth=errwidth,
        )
    ax.set_xlabel(
        xlabel if xlabel is not None else (y if orient == "h" else x),
        fontsize=FONT_LABEL,
    )
    ax.set_ylabel(
        ylabel if ylabel is not None else (x if orient == "h" else y),
        fontsize=FONT_LABEL,
    )
    if title:
        ax.set_title(title, fontsize=FONT_TITLE)
    ax.tick_params(labelsize=FONT_TICK)
    if orient != "h":
        ax.tick_params(axis="x", rotation=xtick_rotation)
    if legend_outside and ax.get_legend() is not None:
        handles, labels = ax.get_legend_handles_labels()
        ax.get_legend().remove()
        ax.legend(
            handles,
            labels,
            bbox_to_anchor=(1.05, 1),
            loc="upper left",
            fontsize=FONT_LEGEND,
            frameon=False,
        )
    sns.despine(ax=ax)
    if save_path:
        ax.get_figure().savefig(FIGURES_DIR / f"{save_path}.svg", bbox_inches="tight")
        ax.get_figure().savefig(
            FIGURES_DIR / f"{save_path}.png", bbox_inches="tight", dpi=200
        )
    return ax


def heatmap_plot(
    data: pd.DataFrame,
    cmap: str = "RdYlGn",
    center: float | None = None,
    vmin: float | None = None,
    vmax: float | None = None,
    annot: bool = False,
    fmt: str = ".2f",
    figsize: tuple = (12, 8),
    xlabel: str | None = None,
    ylabel: str | None = None,
    title: str | None = None,
    save_path: str | None = None,
) -> plt.Axes:
    """Thin wrapper around sns.heatmap with standard project styling."""
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(
        data,
        cmap=cmap,
        center=center,
        vmin=vmin,
        vmax=vmax,
        annot=annot,
        fmt=fmt,
        linewidths=0.3,
        ax=ax,
    )
    ax.set_xlabel(xlabel or "", fontsize=FONT_LABEL)
    ax.set_ylabel(ylabel or "", fontsize=FONT_LABEL)
    if title:
        ax.set_title(title, fontsize=FONT_TITLE)
    ax.tick_params(labelsize=FONT_TICK)
    if save_path:
        fig.savefig(FIGURES_DIR / f"{save_path}.svg", bbox_inches="tight")
        fig.savefig(FIGURES_DIR / f"{save_path}.png", bbox_inches="tight", dpi=200)
    return ax


BEST_COLOR = "#E63946"  # vivid red — reserved for "★ Best" highlights
BEST_HATCH = "////"  # diagonal stripe
BEST_MARKER = "*"
BEST_MARKERSIZE = 400
BEST_EDGECOLOR = "black"
BEST_ALPHA = 0.85
BEST_LABEL = "Best"


def scatter_overlay_best(
    ax: plt.Axes,
    df_best: pd.DataFrame,
    x: str,
    y: str,
    label: str = "Best approaches",
    s: float = BEST_MARKERSIZE,
) -> None:
    """Overlay star markers for best approaches on an existing scatter axes."""
    valid = df_best[[x, y]].dropna()
    if valid.empty:
        return
    ax.scatter(
        valid[x],
        valid[y],
        marker=BEST_MARKER,
        s=s,
        color=BEST_COLOR,
        facecolors="none",
        edgecolors=BEST_EDGECOLOR,
        linewidths=0.7,
        alpha=BEST_ALPHA,
        zorder=10,
        label=label,
    )
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, loc="best", framealpha=0.7, fontsize=FONT_LEGEND)

In [ ]:
# ── Guard: the Figure 3 scoring set must not contain a blind-check metric. ──
# Supplementary File 3 is the published A-K scoring table (see project memory:
# project_nar_supp_file3_authoritative). Comparing against the artefact itself is
# stronger than recomputing the polarity map here.
META_COLS = ["strat", "imp", "method", "post_rm", "status", "compute_time_s"]
BLIND_PREFIXES = ("mk_", "xb_", "pv_")

supp3 = pd.read_csv(SUPP_FILE3_CSV)
scoring_cols = [c for c in supp3.columns if c not in META_COLS]
leaked = [c for c in scoring_cols if c.startswith(BLIND_PREFIXES)]

assert len(scoring_cols) == 87, f"scoring_cols changed: {len(scoring_cols)}"
assert not leaked, f"Blind-check metrics reached the Figure 3 scoring set: {leaked}"
print(
    f"Guard passed: Supplementary File 3 = {supp3.shape[0]} rows x "
    f"{len(scoring_cols)} scoring columns, no mk_/xb_/pv_ leak."
)

In [ ]:
# ── Wide blind-check table ──────────────────────────────────────────────────
# Same metrics_comprehensive CSV the clustermap reads; the L/M/N columns simply live
# there too and are excluded from scoring_cols above.
metrics_wide = pd.read_csv(METRICS_CSV, low_memory=False)
metrics_wide["run_id"] = (
    metrics_wide["strat"]
    + "__"
    + metrics_wide["imp"]
    + "__"
    + metrics_wide["method"]
    + "__post"
    + metrics_wide["post_rm"].map({False: "0", True: "1"})
)
metrics_wide = metrics_wide.set_index("run_id")
assert not metrics_wide.index.duplicated().any(), "duplicate run_id in the wide table"

# ── Analysis set: exactly the filter chain behind Figure 3 ──────────────────
# 1. status == "ok"
# 2. keep one canonical Shambhala representative, renamed 20_shambhala
# 3. drop approaches where >5% of the 5,444 samples come out all-NA
#    (this removes 34_arsyn and 38_harman, 84 rows each)
TOTAL_SAMPLES = 5444
ALLNA_SAMPLE_FRAC = 0.05

analysis = metrics_wide[metrics_wide["status"] == "ok"].copy()
n0 = len(analysis)
analysis = analysis[
    (~analysis["method"].str.startswith("shambhala_"))
    | (analysis["method"] == "shambhala_P0std_Q0std")
].copy()
analysis.loc[analysis["method"] == "shambhala_P0std_Q0std", "method"] = "20_shambhala"
n1 = len(analysis)
# analysis = analysis[
#    analysis["n_samples_allNA"] < TOTAL_SAMPLES * ALLNA_SAMPLE_FRAC].copy()
print(
    f"analysis set: {n0} -> {n1} (Shambhala collapsed) -> {len(analysis)} (NA filter)"
)
assert len(analysis) == 2407, f"analysis set changed: {len(analysis)}"

# ── Blind-check columns, by prefix ──────────────────────────────────────────
# The two Group L families are separated here, so each notebook variant reports exactly
# one of them and the printed counts stay stable when the other family lands.
NARROW_TAG = "_narrow_set"
if NARROW:
    mk_cols = sorted(
        c for c in analysis.columns if c.startswith("mk_") and c.endswith(NARROW_TAG)
    )
else:
    mk_cols = sorted(
        c
        for c in analysis.columns
        if c.startswith("mk_") and not c.endswith(NARROW_TAG)
    )
xb_cols = sorted(c for c in analysis.columns if c.startswith("xb_"))
pv_cols = sorted(c for c in analysis.columns if c.startswith("pv_"))
df_lmn = analysis[META_COLS + mk_cols + xb_cols + pv_cols].copy()

HAS_LMN = bool(mk_cols or xb_cols or pv_cols)
print(
    f"df_lmn: {df_lmn.shape} | mk {len(mk_cols)} | xb {len(xb_cols)} "
    f"| pv {len(pv_cols)}"
)
if HAS_LMN:
    n_with = int(df_lmn[xb_cols[0]].notna().sum()) if xb_cols else 0
    print(f"rows carrying blind-check metrics: {n_with} of {len(df_lmn)}")
else:
    print("\nNo L/M/N columns in the wide table yet — run:")
    print(
        "  python run_metrics_parallel.py --groups L,M,N --only-with-metrics "
        "--skip-shambhala ..."
    )
    print("  python run_metrics_concat.py")
    print("Every cell below degrades to a no-op until then.")

# ── Long-format detail tables ───────────────────────────────────────────────
LONG_SPECS = {
    "gene_long": (GENE_LONG_CSV, ["run_id", "gene", "rho", "n_cohorts"]),
    "cohort_long": (COHORT_LONG_CSV, ["run_id", "cohort", "rho"]),
    "folds_long": (
        FOLDS_LONG_CSV,
        ["run_id", "target", "batch", "n", "n_classes", "f1_macro", "auc"],
    ),
}
long_tables = {}
for _name, (_path, _cols) in LONG_SPECS.items():
    if _path.exists():
        long_tables[_name] = pd.read_csv(_path)
        print(f"  {_name}: {len(long_tables[_name])} rows from {_path.name}")
    else:
        long_tables[_name] = pd.DataFrame(columns=_cols)
        print(f"  {_name}: MISSING ({_path}) — empty table, downstream cells no-op")

# Long tables are produced over all attempts; restrict them to the analysis set so
# every panel below describes the same 2,234 approaches as Figure 3.
for _name in ("gene_long", "cohort_long", "folds_long"):
    _t = long_tables[_name]
    if not _t.empty:
        _before = _t["run_id"].nunique()
        long_tables[_name] = _t[_t["run_id"].isin(df_lmn.index)].copy()
        print(
            f"  {_name}: {_before} -> "
            f"{long_tables[_name]['run_id'].nunique()} runs after analysis-set filter"
        )

gene_long = long_tables["gene_long"]
cohort_long = long_tables["cohort_long"]
folds_long = long_tables["folds_long"]

In [ ]:
# Best-vs-rest labelling, reusing the curated winner list from the v3 notebook.
_top_ids = [
    ("10_mnn", "strict", "S0_no_removal"),
    ("10_mnn", "strict", "H_affymetrix_extended"),
    ("10_mnn", "strict", "D_malignant_only"),
    ("04_sva", "knn", "C_rnaseq_only"),
    ("04_sva", "softimpute", "C_rnaseq_only"),
    ("16_fsqn_r", "knn", "C_rnaseq_only"),
    ("16_fsqn_r", "softimpute", "C_rnaseq_only"),
    ("16_fsqn_r", "strict", "C_rnaseq_only"),
    ("10_mnn", "strict", "J_ff_only"),
    ("16_fsqn_r", "strict", "J_ff_only"),
    ("04_sva", "knn", "K_ffpe_only"),
    ("04_sva", "strict", "K_ffpe_only"),
    ("04_sva", "softimpute", "K_ffpe_only"),
    ("13_fsmvn", "strict", "S0_no_removal"),
    ("33_amdbnorm", "strict", "S0_no_removal"),
]
_top_set = set(_top_ids)

df_lmn = df_lmn.copy()
df_lmn["is_best"] = df_lmn.apply(
    lambda r: (r["method"], r["imp"], r["strat"]) in _top_set and not r["post_rm"],
    axis=1,
)
df_lmn["is_raw"] = df_lmn["method"] == "01_raw"
df_lmn["group"] = np.where(df_lmn["is_best"], "Best (clustermap)", "Rest")

print(
    f"df_lmn: {df_lmn.shape} | best: {int(df_lmn.is_best.sum())} "
    f"| raw baselines: {int(df_lmn.is_raw.sum())}"
)

# Permutation count. A row computed with a different n_perm is not comparable: the
# empirical p-value floor is 1/(n_perm + 1). The 260824 run used 20 permutations
# (floor 0.0476), with three stray rows at other counts.
PV_N_PERM_EXPECTED = 20

if "pv_n_perm" in df_lmn.columns:
    counts = df_lmn["pv_n_perm"].dropna().value_counts()
    print("\npv_n_perm across attempts:")
    print(counts.to_string())
    df_lmn["pv_perm_comparable"] = df_lmn["pv_n_perm"] == PV_N_PERM_EXPECTED
    n_odd = int((df_lmn["pv_n_perm"].notna() & ~df_lmn["pv_perm_comparable"]).sum())
    print(
        f"p-value floor at n_perm = {PV_N_PERM_EXPECTED}: "
        f"{1 / (PV_N_PERM_EXPECTED + 1):.4f}"
    )
    if n_odd:
        print(
            f"{n_odd} rows use a different n_perm and are excluded from the "
            f"p-value panel in section 8."
        )
else:
    df_lmn["pv_perm_comparable"] = False


def _boost_saturation(rgb, delta: float = 0.15):
    h, s, v = mcolors.rgb_to_hsv(rgb[:3])
    return tuple(mcolors.hsv_to_rgb([h, min(1.0, s + delta), v]))


ALL_METHODS = sorted(df_lmn.method.unique())
_tab20 = sns.color_palette("tab20", n_colors=20)
_tab20b = sns.color_palette("tab20b", n_colors=20)
_raw_method_colors = (_tab20 + _tab20b)[: max(len(ALL_METHODS), 25)]

_vivid_method_colors = [_boost_saturation(c) for c in _raw_method_colors]
method_pal = dict(zip(ALL_METHODS, _vivid_method_colors[: len(ALL_METHODS)]))

In [ ]:
# Baseline attachment for the single-matrix groups M and N.
BASELINE_METRICS = [
    c
    for c in [
        "xb_rank_agree",
        "xb_rank_agree_ratio",
        "xb_rank_disagree_diffbio",
        "pv_lobo3_f1_macro_mean",
        "pv_lobo2_f1_macro_mean",
        "pv_lobo3_auc_macro_mean",
        "pv_lobo2_auc_macro_mean",
    ]
    if c in df_lmn.columns
]

# Groups M and N are computed from one matrix, so the before/after comparison is made
# here: 01_raw is itself an attempt in the benchmark, and its value is looked up per
# (strat, imp) pair. The baseline is always 01_raw with post-removal off, so a post1
# row is compared against a baseline with a different sample set — these metrics are
# means over sample pairs and folds rather than sample-matched statistics, so the
# comparison holds, but post1 deltas are approximate and post0/post1 are reported
# separately throughout.
BASELINE_METHOD = "01_raw"

if BASELINE_METRICS:
    baseline = df_lmn[
        (df_lmn["method"] == BASELINE_METHOD) & (~df_lmn["post_rm"].astype(bool))
    ]
    baseline = baseline.set_index([baseline["strat"], baseline["imp"]])
    pair_index = pd.MultiIndex.from_arrays([df_lmn["strat"], df_lmn["imp"]])
    df_lmn["baseline_missing"] = ~pair_index.isin(baseline.index)

    for _m in BASELINE_METRICS:
        _raw = pd.Series(baseline[_m].reindex(pair_index).values, index=df_lmn.index)
        df_lmn[f"{_m}_raw"] = _raw
        df_lmn[f"{_m}_delta"] = df_lmn[_m] - _raw

    print(
        f"baselines attached for {len(BASELINE_METRICS)} metrics from "
        f"{len(baseline)} {BASELINE_METHOD} rows; "
        f"{int(df_lmn['baseline_missing'].sum())} rows have no baseline pair"
    )
else:
    print("No M/N columns yet — baseline attachment skipped.")

In [ ]:
# ── Marker panel annotation ────────────────────────────────────────────────
# Loaded before the gene-set filter below, which needs the in_narrow_set flag.
marker_ann = pd.read_csv(MARKER_ANN_CSV)
marker_ann["is_housekeeping"] = marker_ann["is_housekeeping"].astype(bool)
marker_ann["in_narrow_set"] = marker_ann["in_narrow_set"].astype(bool)
HOUSEKEEPING_GENES = sorted(
    marker_ann.loc[marker_ann["is_housekeeping"], "gene"].unique()
)
NARROW_GENES = sorted(marker_ann.loc[marker_ann["in_narrow_set"], "gene"].unique())
assert len(NARROW_GENES) == 56, f"narrow panel changed: {len(NARROW_GENES)}"
print(
    f"Panel: {marker_ann.gene.nunique()} genes in "
    f"{marker_ann.gene_group.nunique()} groups "
    f"({len(HOUSEKEEPING_GENES)} housekeeping, {len(NARROW_GENES)} narrow-set)."
)

# ── Gene set usable for cross-attempt averages ─────────────────────────────
# Marker coverage differs by batch-removal strategy and imputation, so averaging over
# all genes would compare different gene sets between approaches. min_frac = 1.0
# (present in every single attempt) yields ZERO genes on the 260824 data; 0.9
# reproduces the 178 genes of marker_gene_coverage_audit_260819.csv. Set the constant
# deliberately — an empty COMPLETE_GENES would silently disable the filter downstream.
GENE_MIN_FRAC = 0.9

if NARROW:
    # The narrow panel is itself a coverage-plus-QC selection made on the raw
    # references, so re-gating it on harmonized-attempt coverage would be circular.
    # The gate here is only the panel membership flag.
    COMPLETE_GENES = (
        [] if gene_long.empty else sorted(set(NARROW_GENES) & set(gene_long["gene"]))
    )
    print(
        f"Narrow panel: {len(NARROW_GENES)} genes, "
        f"{len(COMPLETE_GENES)} of them present in the long table."
    )
elif gene_long.empty:
    COMPLETE_GENES = []
    print("gene_long is empty — COMPLETE_GENES = []")
else:
    n_runs = gene_long["run_id"].nunique()
    per_gene_runs = gene_long.dropna(subset=["rho"]).groupby("gene")["run_id"].nunique()
    COMPLETE_GENES = sorted(
        per_gene_runs[per_gene_runs >= GENE_MIN_FRAC * n_runs].index.tolist()
    )
    print(
        f"Genes present in >= {GENE_MIN_FRAC:.0%} of {n_runs} attempts: "
        f"{len(COMPLETE_GENES)} / {gene_long['gene'].nunique()}"
    )
    assert COMPLETE_GENES, (
        f"No gene reaches {GENE_MIN_FRAC:.0%} coverage — lower GENE_MIN_FRAC "
        f"rather than letting the downstream filter silently no-op."
    )

In [ ]:
COMPLETE_GENES = (
    [] if gene_long.empty else sorted(set(NARROW_GENES) & set(gene_long["gene"]))
)
print(
    f"Narrow panel: {len(NARROW_GENES)} genes, "
    f"{len(COMPLETE_GENES)} of them present in the long table."
)

In [ ]:
gene_long

In [ ]:
# ── Cross-check: the wide narrow columns vs the long table ─────────────────────
# The narrow family is a re-aggregation of the same per-gene rho values the long table
# already carries, so these two paths must agree to floating-point noise. If the wide
# narrow columns are absent (the Group L recompute has not run for this DATE_TAG yet),
# the long-table version stands in so the rest of the notebook still works. This runs
# in §1, before §2's figure and §3b's comparison — both read the margin columns.
if NARROW and not gene_long.empty:
    HK = set(HOUSEKEEPING_GENES)
    g = gene_long[gene_long["gene"].isin(NARROW_GENES)].dropna(subset=["rho"])
    from_long = g.groupby("run_id")["rho"].agg(
        mk_rho_mean_all_genes_narrow_set="mean",
        mk_rho_median_all_genes_narrow_set="median",
        mk_rho_min_all_genes_narrow_set="min",
    )
    from_long["mk_rho_frac_genes_above_0.9_narrow_set"] = (
        g.assign(hi=g["rho"] > 0.9).groupby("run_id")["hi"].mean()
    )
    from_long["mk_rho_mean_markers_narrow_set"] = (
        g[~g["gene"].isin(HK)].groupby("run_id")["rho"].mean()
    )
    # Both housekeeping controls are derivable here too: the full panel's controls are in
    # the long table (they are part of the 633-gene panel), and PGK1 is one row of it.
    from_long["mk_rho_mean_housekeeping_narrow_set"] = (
        gene_long[gene_long["gene"].isin(HK)]
        .dropna(subset=["rho"])
        .groupby("run_id")["rho"]
        .mean()
    )
    from_long["mk_rho_mean_PGK1_only_narrow_set"] = (
        gene_long[gene_long["gene"] == "PGK1"]
        .dropna(subset=["rho"])
        .groupby("run_id")["rho"]
        .mean()
    )
    from_long["mk_rho_marker_minus_hk_narrow_set"] = (
        from_long["mk_rho_mean_markers_narrow_set"]
        - from_long["mk_rho_mean_housekeeping_narrow_set"]
    )
    from_long["mk_rho_marker_minus_PGK1_only_narrow_set"] = (
        from_long["mk_rho_mean_markers_narrow_set"]
        - from_long["mk_rho_mean_PGK1_only_narrow_set"]
    )

    col = "mk_rho_mean_all_genes_narrow_set"
    if col in df_lmn.columns and df_lmn[col].notna().any():
        both = df_lmn[[col]].join(from_long[[col]], rsuffix="_long").dropna()
        d = (both[col] - both[f"{col}_long"]).abs()
        print(
            f"Wide vs long narrow mean rho: n = {len(both)}, "
            f"max |diff| = {d.max():.2e}"
        )
        assert d.max() < 1e-6, (
            "The wide mk_*_narrow_set columns disagree with the long table. One of the "
            "two is stale — regenerate the metric tables before reading any figure."
        )
    else:
        print(
            "No mk_*_narrow_set columns in this snapshot — falling back to the "
            "long-table aggregates."
        )
        for c in from_long.columns:
            df_lmn[c] = from_long[c]
        mk_cols = sorted(set(mk_cols) | set(from_long.columns))

## §2 — Best vs rest: do the clustermap winners also win the blind check?

Each panel compares the fifteen curated best approaches against every other attempt.
`xb_rank_agree_ratio` is the headline: it is the within-matrix margin between
same-biology and different-biology cross-batch agreement, so it cannot be inflated by a
method that simply collapses all samples toward one profile.

In [ ]:
df_lmn['xb_rank_agree'][df_lmn.group == 'Best (clustermap)']

In [ ]:
df_lmn['xb_rank_agree'][df_lmn.group != 'Best (clustermap)'].median()

In [ ]:
HEADLINE = [
    (f"mk_rho_mean_all_genes{MK_SFX}", "Group L\nmean per-gene $\\rho$"),
    (f"mk_rho_marker_minus_hk{MK_SFX}", "Group L\nmarker $-$ housekeeping"),
    ("mk_rho_marker_minus_PGK1_only_narrow_set", "Group L\nmarker $-$ PGK1 only"),
    ("xb_rank_agree", "Group M\ncross-batch agreement"),
    ("xb_rank_agree_ratio", "Group M\nbiology margin"),
    ("xb_rank_agree_ratio_delta", "Group M\nmargin gain vs raw"),
    ("pv_lobo3_f1_macro_mean", "Group N\nLOBO macro-F1 (3-class)"),
    ("pv_lobo2_auc_macro_mean", "Group N\nLOBO AUC (FL vs DLBCL)"),
]
HEADLINE = [(c, lab) for c, lab in HEADLINE if c in df_lmn.columns]


def best_vs_rest_panel(ax, col, label, data):
    """Boxplot of best vs rest with points overlaid, annotated with a Mann-Whitney p."""
    sub = data[[col, "group"]].dropna()
    if sub.empty or sub["group"].nunique() < 2:
        ax.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax.transAxes)
        ax.set_axis_off()
        return
    order = ["Best (clustermap)", "Rest"]
    sns.boxplot(
        data=sub,
        x="group",
        y=col,
        order=order,
        ax=ax,
        width=0.55,
        showfliers=False,
        palette=[COL_BEST, COL_REST],
    )
    sns.stripplot(
        data=sub,
        x="group",
        y=col,
        order=order,
        ax=ax,
        size=1.6,
        color="black",
        alpha=0.35,
        jitter=0.28,
    )
    a = sub.loc[sub["group"] == order[0], col]
    b = sub.loc[sub["group"] == order[1], col]
    if len(a) > 2 and len(b) > 2:
        u, p = mannwhitneyu(a, b, alternative="two-sided")
        # Rank-biserial correlation: a scale-free effect size for Mann-Whitney.
        rbc = 2 * u / (len(a) * len(b)) - 1
        ax.set_title(f"{label}\np = {p:.2g}, rbc = {rbc:+.2f}", pad=3)
    else:
        ax.set_title(label, pad=3)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(axis="x", labelrotation=20)


if HEADLINE:
    n = len(HEADLINE)
    ncol = min(4, n)
    nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(1.9 * ncol, 2.3 * nrow))
    axes = np.atleast_1d(axes).ravel()
    for ax, (col, label) in zip(axes, HEADLINE):
        best_vs_rest_panel(ax, col, label, df_lmn[~df_lmn["is_raw"]])
    for ax in axes[n:]:
        ax.set_axis_off()
    fig.suptitle("Blind check: curated best approaches vs all others", y=1.01)
    fig.tight_layout()
    save_figure(fig, "fig5a_best_vs_rest", FIG_DIR)
    plt.show()
else:
    print("No blind-check columns yet — panel skipped.")

### Individual box plots by methods

In [ ]:
df_lmn['pv_lobo2_auc_macro_mean'][df_lmn["group"] == 'Best (clustermap)'].median()

In [ ]:
df_lmn["group"]

In [ ]:
def plot_palette(palette, title=None, filename="palette.svg"):
    """
    Display a color palette as a legend-style swatch figure.

    Parameters
    ----------
    palette : dict
        Mapping of label → color (any matplotlib-compatible color format).
    title : str, optional
        Figure title shown above the swatches.
    """
    fig, ax = plt.subplots(figsize=(4, max(2.5, len(palette) * 0.1)))
    ax.axis("off")
    handles = [
        mpatches.Patch(color=color, label=str(name)) for name, color in palette.items()
    ]
    ax.legend(
        handles=handles,
        loc="center",
        ncol=1,
        fontsize=7.5,
        frameon=False,
        labelspacing=0.1,
        title=title,
    )
    plt.tight_layout()

    plt.savefig(filename, transparent=True)
    plt.show()


plot_palette(method_pal, title="method_pal", filename="./figures/method_pal.svg")

In [ ]:
df_lmn

In [ ]:
    fig, ax = plt.subplots(1, 1, figsize=(2, 1.5))

    col = 'mk_rho_mean_all_genes_narrow_set'
    sub = df_lmn[[col, "group", 'method']].dropna()

    order = ["Best (clustermap)", "Rest"]
    sns.boxplot(data=sub, x="group", y=col, order=order, ax=ax, width=0.55,
                showfliers=False,
                palette=[
                    '#F9CED1',
                    '#C6CCD6'
                ])
    sns.stripplot(data=sub, x="group", y=col, order=order, ax=ax, size=1.6,
                  hue="method", palette=method_pal, alpha=1, jitter=0.28, legend=False)
    a = sub.loc[sub["group"] == order[0], col]
    b = sub.loc[sub["group"] == order[1], col]
    if len(a) > 2 and len(b) > 2:
        u, p = mannwhitneyu(a, b, alternative="two-sided")
        # Rank-biserial correlation: a scale-free effect size for Mann-Whitney.
        rbc = 2 * u / (len(a) * len(b)) - 1
        ax.set_title(f"p = {p:.2g}, rbc = {rbc:+.2f}", pad=3, fontsize=7.5)

    #ax.tick_params(axis="x", labelrotation=20)

In [ ]:
FIG_DIR

In [ ]:
df_lmn[f"mk_rho_mean_all_genes{MK_SFX}"][df_lmn.group == "Best (clustermap)"]

In [ ]:
HEADLINE = [
    (f"mk_rho_mean_all_genes{MK_SFX}", "Group L\nmean per-gene $\\rho$"),
    (f"mk_rho_marker_minus_hk{MK_SFX}", "Group L\nmarker $-$ housekeeping"),
    ("mk_rho_marker_minus_PGK1_only_narrow_set", "Group L\nmarker $-$ PGK1 only"),
    ("xb_rank_agree", "Group M\ncross-batch agreement"),
    ("xb_rank_agree_ratio", "Group M\nbiology margin"),
    ("xb_rank_agree_ratio_delta", "Group M\nmargin gain vs raw"),
    ("pv_lobo3_f1_macro_mean", "Group N\nLOBO macro-F1 (3-class)"),
    ("pv_lobo2_auc_macro_mean", "Group N\nLOBO AUC (FL vs DLBCL)"),
]
HEADLINE = [(c, lab) for c, lab in HEADLINE if c in df_lmn.columns]


def best_vs_rest_panel(ax, col, label, data):
    """Boxplot of best vs rest with points overlaid, annotated with a Mann-Whitney p."""

    sub = data[[col, "group", "method"]].dropna()

    order = ["Best (clustermap)", "Rest"]
    sns.boxplot(
        data=sub,
        x="group",
        y=col,
        order=order,
        ax=ax,
        width=0.55,
        showfliers=False,
        palette=["#F9CED1", "#C6CCD6"],
    )
    sns.stripplot(
        data=sub,
        x="group",
        y=col,
        order=order,
        ax=ax,
        size=1.6,
        hue="method",
        palette=method_pal,
        alpha=1,
        jitter=0.28,
        legend=False,
    )
    a = sub.loc[sub["group"] == order[0], col]
    b = sub.loc[sub["group"] == order[1], col]
    if len(a) > 2 and len(b) > 2:
        u, p = mannwhitneyu(a, b, alternative="two-sided")
        # Rank-biserial correlation: a scale-free effect size for Mann-Whitney.
        rbc = 2 * u / (len(a) * len(b)) - 1
        ax.set_title(f"{label}\np = {p:.2g}, rbc = {rbc:+.2f}", pad=3)
    ax.set_xlabel("")
    ax.set_ylabel("")


if HEADLINE:
    n = len(HEADLINE)
    ncol = min(4, n)
    nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(2 * ncol, 2 * nrow))
    axes = np.atleast_1d(axes).ravel()
    for ax, (col, label) in zip(axes, HEADLINE):
        best_vs_rest_panel(ax, col, label, df_lmn[~df_lmn["is_raw"]])
    for ax in axes[n:]:
        ax.set_axis_off()
    fig.suptitle("Blind check: curated best approaches vs all others", y=1.01)
    fig.tight_layout()
    save_figure(fig, "fig5a_best_vs_rest_with_methods", FIG_DIR)
    plt.show()
else:
    print("No blind-check columns yet — panel skipped.")

In [ ]:
mcolors.to_hex(mcolors.to_rgba("#F29CA2", alpha=0.01))

In [ ]:
COL_BEST, COL_REST

## §3 — Group L by gene group and cell type

Housekeeping genes are the negative control: they sit at the same expression level in
every sample, so their within-cohort ranks carry little signal and a method that genuinely
reorders samples drives their $\rho$ down. The wanted signature is **high $\rho$ on
biologically variable markers, lower $\rho$ on housekeeping genes**. A method applying a
per-batch monotone transform keeps both at ~1.0, so read the two together, never the
housekeeping value alone.

In [ ]:
def gene_long_annotated(genes_subset=None):
    """gene_long joined to the panel annotation and the best/rest label."""
    if gene_long.empty:
        return pd.DataFrame()
    g = gene_long.copy()
    if genes_subset:
        g = g[g["gene"].isin(genes_subset)]
    ann = marker_ann[["gene", "gene_group", "cell_type", "is_housekeeping"]]
    g = g.merge(ann, on="gene", how="left")
    meta = df_lmn[["method", "imp", "strat", "post_rm", "is_best", "group"]]
    return g.merge(meta, left_on="run_id", right_index=True, how="left")


gl = gene_long_annotated(COMPLETE_GENES)
print(f"annotated gene-level rows: {len(gl)}")

In [ ]:
if not gl.empty:
    by_ct = gl[gl["cell_type"].notna() & (gl["cell_type"] != "none")]
    order = (
        by_ct.groupby("cell_type")["rho"].median().sort_values(ascending=False).index
    )
    hk_med = gl.loc[gl["is_housekeeping"], "rho"].median()

    fig, axes = plt.subplots(
        1, 2, figsize=(7.2, 2.8), gridspec_kw={"width_ratios": [1.6, 1]}
    )

    sns.boxplot(
        data=by_ct,
        y="cell_type",
        x="rho",
        order=order,
        ax=axes[0],
        showfliers=False,
        color=COL_BEST,
        width=0.6,
    )
    if np.isfinite(hk_med):
        axes[0].axvline(
            hk_med,
            color=COL_HK,
            ls="--",
            lw=1,
            label=f"housekeeping median = {hk_med:.2f}",
        )
        axes[0].legend(loc="lower left", frameon=False)
    axes[0].set_xlabel("per-gene Spearman $\\rho$ (before vs after)")
    axes[0].set_ylabel("")
    axes[0].set_title("A  By associated cell type", loc="left")

    panel_med = (
        gl.groupby(["gene_group", "is_housekeeping"])["rho"]
        .median()
        .reset_index()
        .sort_values("rho", ascending=False)
    )
    top = pd.concat([panel_med.head(12), panel_med[panel_med["is_housekeeping"]]])
    colors = [COL_HK if hk else COL_BEST for hk in top["is_housekeeping"]]
    axes[1].barh(top["gene_group"], top["rho"], color=colors, height=0.7)
    axes[1].invert_yaxis()
    axes[1].set_xlabel("median $\\rho$")
    axes[1].set_title("B  By signature (housekeeping in grey)", loc="left")

    fig.tight_layout()
    save_figure(fig, "fig5b_group_l_by_gene_group", FIG_DIR)
    plt.show()
else:
    print("gene_long is empty — panel skipped.")

### §3b — Which housekeeping control is less noisy?

`mk_rho_marker_minus_hk_narrow_set` subtracts the full housekeeping panel; only 6 of
those 15 genes clear the coverage bar the narrow markers had to clear, so its control set
changes composition between attempts. `mk_rho_marker_minus_PGK1_only_narrow_set`
subtracts `PGK1` alone: coverage-matched, but a single gene.

Three readings, all on the same rows:

1. **Dispersion within a fixed `(strat, imp)` pair.** Attempts inside one pair share a
   sample set and a gene set, so spread there is method effect plus metric noise. The
   noisier control inflates it.
2. **Dispersion of the control term itself** — `sd` of
   `mk_rho_mean_housekeeping_narrow_set` versus `mk_rho_mean_PGK1_only_narrow_set`.
3. **Discriminative power** — the best-vs-rest rank-biserial correlation of each margin.
   A noisier metric buys the same separation at a lower effect size.

A margin that wins on 1 and 3 is the one to quote in the manuscript.

In [ ]:
# Fig 5f — the two housekeeping controls, compared on noise and on discrimination.
PAIR = [
    ("mk_rho_marker_minus_hk_narrow_set", "full housekeeping panel"),
    ("mk_rho_marker_minus_PGK1_only_narrow_set", "PGK1 only"),
]
PAIR = [(c, lab) for c, lab in PAIR if c in df_lmn.columns]

if NARROW and len(PAIR) == 2:
    d = df_lmn[~df_lmn["is_raw"]].copy()
    rows = []
    for col, lab in PAIR:
        ser = d[col].dropna()
        # Within-pair spread: mean over (strat, imp) of the per-pair SD.
        within = d.dropna(subset=[col]).groupby(["strat", "imp"])[col].std().mean()
        a = d.loc[d["is_best"], col].dropna()
        b = d.loc[~d["is_best"], col].dropna()
        if len(a) > 2 and len(b) > 2:
            u, pv = mannwhitneyu(a, b, alternative="two-sided")
            rbc = 2 * u / (len(a) * len(b)) - 1
        else:
            pv, rbc = np.nan, np.nan
        rows.append(
            {
                "control": lab,
                "metric": col,
                "n": len(ser),
                "mean": ser.mean(),
                "sd": ser.std(),
                "iqr": ser.quantile(0.75) - ser.quantile(0.25),
                "mad": (ser - ser.median()).abs().median(),
                "within_pair_sd": within,
                "best_vs_rest_rbc": rbc,
                "best_vs_rest_p": pv,
            }
        )
    noise_cmp = pd.DataFrame(rows)
    print(noise_cmp.round(4).to_string(index=False))
    noise_cmp.to_csv(
        FIG_DIR / f"T5f_hk_control_noise_comparison{MK_SFX}.csv", index=False
    )

    quieter = noise_cmp.loc[noise_cmp["within_pair_sd"].idxmin(), "control"]
    sharper = noise_cmp.loc[noise_cmp["best_vs_rest_rbc"].abs().idxmax(), "control"]
    print(f"\nLower within-(strat, imp) SD : {quieter}")
    print(f"Larger best-vs-rest effect   : {sharper}")
    if quieter == sharper:
        print(f"=> quote the {quieter} margin in the manuscript.")
    else:
        print("=> the two readings disagree; report both margins and say so.")

    fig, axes = plt.subplots(1, 3, figsize=(8.4, 2.8))
    # A  distribution of each margin
    long = d[[c for c, _ in PAIR]].melt(var_name="metric", value_name="margin").dropna()
    long["control"] = long["metric"].map(dict((c, l) for c, l in PAIR))
    sns.violinplot(
        data=long,
        x="control",
        y="margin",
        ax=axes[0],
        palette=[COL_BEST, COL_HK],
        cut=0,
        inner="quartile",
    )
    axes[0].set_title("A  Margin distribution", loc="left")
    axes[0].set_xlabel("")
    axes[0].tick_params(axis="x", labelrotation=15)
    # B  within-pair SD side by side
    axes[1].bar(
        noise_cmp["control"],
        noise_cmp["within_pair_sd"],
        color=[COL_BEST, COL_HK],
        width=0.6,
    )
    axes[1].set_title("B  Within-(strat, imp) SD", loc="left")
    axes[1].tick_params(axis="x", labelrotation=15)
    # C  agreement between the two margins
    sub = d[[c for c, _ in PAIR]].dropna()
    axes[2].scatter(sub.iloc[:, 0], sub.iloc[:, 1], s=3, alpha=0.35, color=COL_REST)
    lim = [float(sub.min().min()), float(sub.max().max())]
    axes[2].plot(lim, lim, ls="--", lw=0.8, color="k")
    r = sub.corr(method="spearman").iloc[0, 1]
    axes[2].set_xlabel("full housekeeping")
    axes[2].set_ylabel("PGK1 only")
    axes[2].set_title(f"C  Agreement, $\\rho$ = {r:.2f}", loc="left")

    fig.tight_layout()
    save_figure(fig, "fig5f_hk_control_comparison", FIG_DIR)
    plt.show()
elif NARROW:
    print(
        "Both narrow margins are needed for the comparison — "
        f"found {[c for c, _ in PAIR]}."
    )

## §4 — Group L by individual gene

Some genes keep their profile and others do not. The heatmap is restricted to the genes
measured in at least `GENE_MIN_FRAC` of the attempts (§1), so the columns are comparable.
No gene is present in literally every attempt, which is why the threshold is a constant
rather than a strict intersection.

In [ ]:
if not gl.empty:
    piv = gl[~gl["method"].eq("01_raw")].pivot_table(
        index="gene", columns="run_id", values="rho"
    )
    gene_order = piv.mean(axis=1).sort_values(ascending=False).index
    piv = piv.loc[gene_order]

    fig = plt.figure(figsize=(7.2, 5.4))
    gs = gridspec.GridSpec(1, 2, width_ratios=[2.2, 1], wspace=0.35)

    ax0 = fig.add_subplot(gs[0])
    sns.heatmap(
        piv,
        ax=ax0,
        cmap="RdYlBu_r",
        vmin=0,
        vmax=1,
        cbar_kws={"label": "$\\rho$"},
        xticklabels=False,
        yticklabels=max(1, len(piv) // 45),
    )
    ax0.set_xlabel(f"harmonization attempts (n = {piv.shape[1]})")
    ax0.set_ylabel("")
    ax0.set_title("A  Gene x approach preservation", loc="left")

    ax1 = fig.add_subplot(gs[1])
    means = piv.mean(axis=1)
    tail = pd.concat([means.head(15), means.tail(15)])
    is_hk = tail.index.isin(gl.loc[gl["is_housekeeping"], "gene"].unique())
    ax1.hlines(
        range(len(tail)),
        0,
        tail.values,
        color=[COL_HK if h else COL_BEST for h in is_hk],
        lw=1.2,
    )
    ax1.plot(
        tail.values, range(len(tail)), "o", ms=2.5, color="black", mfc="none", lw=0
    )
    ax1.set_yticks(range(len(tail)))
    ax1.set_yticklabels(tail.index)
    ax1.invert_yaxis()
    ax1.axhline(14.5, color="grey", lw=0.6, ls=":")
    ax1.set_xlabel("mean $\\rho$ across attempts")
    ax1.set_title("B  Best- and worst-preserved genes", loc="left")

    save_figure(fig, "fig5c_group_l_by_gene", FIG_DIR)
    plt.show()
else:
    print("gene_long is empty — panel skipped.")

## §5 — Marker panel coverage

Coverage differs by batch-removal strategy and imputation. The clinical markers the
colleagues named (CD20 = `MS4A1`, CD3 = `CD3D`/`CD3E`, CD10 = `MME`, CD23 = `FCER2`,
CD79A/B, PD-1 = `PDCD1`) are absent from the strict full-coverage intersection but present
in roughly two-thirds of `(strat, imp)` pairs via the imputed matrices. The article must
report coverage per gene rather than claiming the markers were unavailable.

In [ ]:
if COVERAGE_AUDIT_CSV.exists():
    audit = pd.read_csv(COVERAGE_AUDIT_CSV).sort_values(
        "n_pairs_present", ascending=False
    )
    n_pairs = int(audit["n_pairs_present"].max())

    KEY = [
        "MS4A1",
        "CD3D",
        "CD3E",
        "MME",
        "FCER2",
        "CR2",
        "CD79A",
        "CD79B",
        "BCL2",
        "BCL6",
        "MKI67",
        "IRF4",
        "MYC",
        "CD4",
        "CD8A",
        "CD68",
        "CD163",
        "FOXP3",
        "PDCD1",
        "ACTB",
        "GAPDH",
        "B2M",
    ]
    key = audit[audit["gene"].isin(KEY)].sort_values("n_pairs_present")

    fig, axes = plt.subplots(
        1, 2, figsize=(7.2, 2.9), gridspec_kw={"width_ratios": [1, 1.1]}
    )
    axes[0].hist(
        audit["n_pairs_present"],
        bins=np.arange(-0.5, n_pairs + 1.5, 1),
        color=COL_REST,
        edgecolor="white",
        lw=0.3,
    )
    axes[0].set_xlabel(
        f"number of (strat, imp) pairs containing the gene (max {n_pairs})"
    )
    axes[0].set_ylabel("genes")
    axes[0].set_title(f"A  Panel coverage, {len(audit)} genes", loc="left")

    axes[1].barh(key["gene"], key["n_pairs_present"], color=COL_BEST, height=0.7)
    axes[1].axvline(n_pairs, color="black", ls="--", lw=0.8, label="all pairs")
    axes[1].set_xlabel("pairs present")
    axes[1].legend(frameon=False, loc="lower right")
    axes[1].set_title("B  Clinical / IHC markers", loc="left")

    fig.tight_layout()
    save_figure(fig, "fig5d_panel_coverage", FIG_DIR)
    plt.show()

    print(
        f"\nComplete in all {n_pairs} pairs: "
        f"{int((audit.n_pairs_present == n_pairs).sum())} genes"
    )
    print(
        f"Absent everywhere: "
        f"{', '.join(audit.loc[audit.n_pairs_present == 0, 'gene'])}"
    )
else:
    print(
        f"{COVERAGE_AUDIT_CSV} not found — regenerate with the Phase 0 coverage audit."
    )

In [ ]:
_cov_col = f"mk_n_panel_genes_used{MK_SFX}"
if _cov_col in df_lmn.columns:
    cov = df_lmn.groupby(["strat", "imp"])[_cov_col].max().unstack().sort_index()
    fig, ax = plt.subplots(figsize=(4.2, 4.0))
    sns.heatmap(
        cov,
        annot=True,
        fmt=".0f",
        cmap="YlGnBu",
        ax=ax,
        cbar_kws={"label": "panel genes resolved"},
        annot_kws={"size": FONT_SIZE - 1},
    )
    ax.set_title("Panel genes available per strategy x imputation", loc="left")
    ax.set_xlabel("")
    ax.set_ylabel("")
    fig.tight_layout()
    save_figure(fig, "fig5e_coverage_by_strat_imp", FIG_DIR)
    plt.show()
else:
    print(f"{_cov_col} not in the table yet — panel skipped.")

## §6 — Group M: cross-batch rank agreement

Panel A pairs each attempt's agreement with its unharmonized `01_raw` baseline, so the
slope is the gain harmonization delivered. Panel B is the check that matters: the
same-biology gain must exceed the different-biology gain, otherwise the method is
collapsing every sample toward a common profile rather than removing batch effect.

In [ ]:
if "xb_rank_agree" in df_lmn.columns and "xb_rank_agree_raw" in df_lmn.columns:
    d = df_lmn[~df_lmn["is_raw"]].dropna(subset=["xb_rank_agree", "xb_rank_agree_raw"])
    fig, axes = plt.subplots(1, 3, figsize=(7.6, 2.9))

    # A — raw to harmonized, one line per attempt.
    for _, row in d.iterrows():
        axes[0].plot(
            [0, 1],
            [row["xb_rank_agree_raw"], row["xb_rank_agree"]],
            color=COL_BEST if row["is_best"] else COL_REST,
            alpha=0.9 if row["is_best"] else 0.12,
            lw=1.1 if row["is_best"] else 0.5,
            zorder=3 if row["is_best"] else 1,
        )
    axes[0].set_xticks([0, 1])
    axes[0].set_xticklabels(["01_raw", "harmonized"])
    axes[0].set_ylabel("cross-batch agreement (same biology)")
    axes[0].set_title("A  Gain per attempt", loc="left")
    axes[0].plot([], [], color=COL_BEST, lw=1.1, label="best")
    axes[0].plot([], [], color=COL_REST, lw=0.5, label="rest")
    axes[0].legend(frameon=False, loc="best")

    # B — same-biology vs different-biology gain; y = x means no biology-specific gain.
    if (
        "xb_rank_disagree_diffbio_delta" in d.columns
        and "xb_rank_agree_delta" in d.columns
    ):
        axes[1].scatter(
            d["xb_rank_disagree_diffbio_delta"],
            d["xb_rank_agree_delta"],
            s=5,
            c=np.where(d["is_best"], COL_BEST, COL_REST),
            alpha=0.75,
            linewidths=0,
        )
        lims = np.nanpercentile(
            np.concatenate(
                [
                    d["xb_rank_disagree_diffbio_delta"].values,
                    d["xb_rank_agree_delta"].values,
                ]
            ),
            [1, 99],
        )
        axes[1].plot(lims, lims, ls="--", color="black", lw=0.8)
        axes[1].set_xlabel("different-biology gain")
        axes[1].set_ylabel("same-biology gain")
        axes[1].set_title("B  Above the diagonal = real\nbatch correction", loc="left")

    # C — the margin itself.
    sub = d[["xb_rank_agree_ratio", "group"]].dropna()
    if not sub.empty:
        sns.boxplot(
            data=sub,
            x="group",
            y="xb_rank_agree_ratio",
            ax=axes[2],
            order=["Best (clustermap)", "Rest"],
            showfliers=False,
            palette=[COL_BEST, COL_REST],
            width=0.55,
        )
        raw_med = df_lmn.loc[df_lmn["is_raw"], "xb_rank_agree_ratio"].median()
        if np.isfinite(raw_med):
            axes[2].axhline(
                raw_med, color=COL_RAW, ls="--", lw=1, label="01_raw median"
            )
            axes[2].legend(frameon=False)
        axes[2].set_xlabel("")
        axes[2].set_ylabel("biology margin")
        axes[2].set_title("C  Margin vs raw", loc="left")
        axes[2].tick_params(axis="x", labelrotation=20)

    fig.tight_layout()
    save_figure(fig, "fig6a_group_m", FIG_DIR)
    plt.show()
else:
    print("Group M columns not in the table yet — panel skipped.")

In [ ]:
df_lmn["group"].value_counts()

### Additional plots

In [ ]:
df_lmn.columns[df_lmn.columns.str.startswith("xb_rank")]

In [ ]:
x_col='xb_rank_disagree_diffbio'
y_col='xb_rank_agree'
df_lmn_corr = df_lmn[[x_col, y_col]].dropna()


pearsonr(df_lmn_corr[x_col], df_lmn_corr[y_col])

In [ ]:
(df_lmn['xb_rank_agree_delta'] - df_lmn['xb_rank_disagree_diffbio_delta']).dropna().sort_values()[-20:]

In [ ]:
df_lmn.T['K_ffpe_only__knn__06_combat_seq__post0'].T[['xb_rank_agree_delta', 'xb_rank_disagree_diffbio_delta']]

In [ ]:
    hue_col='method'
    x_col='xb_rank_disagree_diffbio_delta'
    y_col='xb_rank_agree_delta'

    ax = scatter_plot(
        df_lmn,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=method_pal,
        alpha=0.75,
        s=20,
        figsize=(2.2, 2.2),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_lmn[df_lmn['group'] == 'Best (clustermap)'], x=x_col, y=y_col)
    ax.legend().set_visible(False)
    ax.get_figure().savefig( f"../figures_for_article/current_figures_tables_for_article_260819/a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig( f"../figures_for_article/current_figures_tables_for_article_260819/a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
    hue_col='method'
    x_col='xb_rank_disagree_diffbio'
    y_col='xb_rank_agree'

    ax = scatter_plot(
        df_lmn,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=method_pal,
        alpha=0.75,
        s=20,
        figsize=(2.2, 2.2),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_lmn[df_lmn['group'] == 'Best (clustermap)'], x=x_col, y=y_col)
    ax.legend().set_visible(False)
    ax.get_figure().savefig( f"../figures_for_article/current_figures_tables_for_article_260819/a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig( f"../figures_for_article/current_figures_tables_for_article_260819/a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
    second_col = "xb_rank_agree"
    first_col = "xb_rank_agree_raw"

    d = df_lmn[~df_lmn["is_raw"]].dropna(subset=[first_col, second_col])
    fig, ax = plt.subplots(figsize=(3, 3))

    # A — raw to harmonized, one line per attempt.
    for _, row in d.iterrows():
        ax.plot([0, 1], [row[first_col], row[second_col]],
                     color=method_pal.get(row['method']) if row["is_best"] else COL_REST,
                     alpha=0.9 if row["is_best"] else 0.12,
                     lw=1.1 if row["is_best"] else 0.5, zorder=3 if row["is_best"] else 1)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["01_raw", "harmonized"])
    ax.set_ylabel("cross-batch agreement (same biology)")
    ax.set_title("A  Gain per attempt", loc="left")
    ax.plot([], [], color=COL_BEST, lw=1.1, label="best")
    ax.plot([], [], color=COL_REST, lw=0.5, label="rest")
    ax.legend(frameon=False, loc="best")
    ax.get_figure().savefig( f"../figures_for_article/current_figures_tables_for_article_260819/transition_plot_{first_col}_to_{second_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig( f"../figures_for_article/current_figures_tables_for_article_260819/transition_plot_{first_col}_to_{second_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
# Per-diagnosis agreement: run_metrics_concat pivots the nested dict into
# xb_rank_agree_{diagnosis} columns.
diag_cols = [
    c
    for c in df_lmn.columns
    if c.startswith("xb_rank_agree_")
    and not c.endswith(("_raw", "_delta", "_median", "_ratio"))
    and c not in ("xb_rank_agree_by_diagnosis",)
]
if diag_cols:
    d = df_lmn[~df_lmn["is_raw"]]
    long = (
        d[diag_cols + ["group"]]
        .melt(id_vars="group", var_name="diagnosis", value_name="agreement")
        .dropna()
    )
    long["diagnosis"] = long["diagnosis"].str.replace("xb_rank_agree_", "", regex=False)
    keep = long["diagnosis"].value_counts()
    long = long[long["diagnosis"].isin(keep[keep > 20].index)]
    if not long.empty:
        order = (
            long.groupby("diagnosis")["agreement"]
            .median()
            .sort_values(ascending=False)
            .index
        )
        fig, ax = plt.subplots(figsize=(6.4, 2.7))
        sns.boxplot(
            data=long,
            x="diagnosis",
            y="agreement",
            hue="group",
            order=order,
            ax=ax,
            showfliers=False,
            palette=[COL_BEST, COL_REST],
            width=0.7,
        )
        ax.tick_params(axis="x", labelrotation=35)
        ax.set_xlabel("")
        ax.set_ylabel("cross-batch agreement")
        ax.set_title("Group M by diagnosis", loc="left")
        ax.legend(frameon=False, title="")
        fig.tight_layout()
        save_figure(fig, "fig6b_group_m_by_diagnosis", FIG_DIR)
        plt.show()
else:
    print("No per-diagnosis columns yet — panel skipped.")

## §7 — Group N: leave-one-batch-out prediction

F1 uses every fold, including single-class test batches: a predictor facing a DLBCL-only
batch should label as many of its samples DLBCL as it can, which is the honest deployment
test. AUC is restricted to folds whose test batch holds at least two classes. Both fold
counts are annotated, because neither mean is interpretable without them.

In [ ]:
if "pv_lobo3_f1_macro_mean" in df_lmn.columns:
    d = df_lmn[~df_lmn["is_raw"]]
    fig, axes = plt.subplots(1, 3, figsize=(7.6, 2.9))

    for ax, (col, perm_col, label) in zip(
        axes,
        [
            (
                "pv_lobo3_f1_macro_mean",
                "pv_lobo3_f1_macro_perm_mean",
                "3-class macro-F1",
            ),
            (
                "pv_lobo2_f1_macro_mean",
                "pv_lobo2_f1_macro_perm_mean",
                "FL vs DLBCL macro-F1",
            ),
            (
                "pv_lobo2_auc_macro_mean",
                "pv_lobo2_auc_macro_perm_mean",
                "FL vs DLBCL AUC",
            ),
        ],
    ):
        if col not in d.columns:
            ax.set_axis_off()
            continue
        sub = d[[col, "group"]].dropna()
        if sub.empty:
            ax.set_axis_off()
            continue
        sns.boxplot(
            data=sub,
            x="group",
            y=col,
            ax=ax,
            order=["Best (clustermap)", "Rest"],
            showfliers=False,
            palette=[COL_BEST, COL_REST],
            width=0.55,
        )
        raw_med = df_lmn.loc[df_lmn["is_raw"], col].median()
        if np.isfinite(raw_med):
            ax.axhline(raw_med, color=COL_RAW, ls="--", lw=1, label="01_raw")
        if perm_col in d.columns:
            lo, hi = d[perm_col].quantile([0.05, 0.95])
            if np.isfinite(lo) and np.isfinite(hi):
                ax.axhspan(
                    lo, hi, color=COL_PERM, alpha=0.7, zorder=0, label="permuted null"
                )
        ax.legend(frameon=False, fontsize=FONT_SIZE - 1, loc="lower left")
        ax.set_xlabel("")
        ax.set_ylabel(label)
        ax.tick_params(axis="x", labelrotation=20)

    n_f = (
        df_lmn["pv_lobo3_n_folds"].median() if "pv_lobo3_n_folds" in df_lmn else np.nan
    )
    n_mc = (
        df_lmn["pv_lobo3_n_folds_multiclass"].median()
        if "pv_lobo3_n_folds_multiclass" in df_lmn
        else np.nan
    )
    fig.suptitle(
        f"Leave-one-batch-out prediction (median {n_f:.0f} folds, "
        f"{n_mc:.0f} multi-class)",
        y=1.03,
    )
    fig.tight_layout()
    save_figure(fig, "fig6c_group_n", FIG_DIR)
    plt.show()
else:
    print("Group N columns not in the table yet — panel skipped.")

In [ ]:
if not folds_long.empty:
    meta = df_lmn[["method", "is_best", "group", "is_raw"]]
    f = folds_long.merge(meta, left_on="run_id", right_index=True, how="left")
    f3 = f[(f["target"] == "3class") & (~f["is_raw"].fillna(False))]
    if not f3.empty:
        order = f3.groupby("batch")["f1_macro"].median().sort_values().index
        fig, ax = plt.subplots(figsize=(7.2, 3.2))
        sns.stripplot(
            data=f3,
            x="f1_macro",
            y="batch",
            order=order,
            hue="group",
            ax=ax,
            size=1.8,
            alpha=0.5,
            dodge=False,
            palette=[COL_BEST, COL_REST],
        )
        single = f3[f3["n_classes"] == 1]["batch"].unique()
        labels = [f"{b}  *" if b in single else b for b in order]
        ax.set_yticklabels(labels)
        ax.set_xlabel("fold macro-F1 (3-class)")
        ax.set_ylabel("")
        ax.set_title(
            "Per-batch difficulty  (* = single-class batch, no AUC)", loc="left"
        )
        ax.legend(frameon=False, title="")
        fig.tight_layout()
        save_figure(fig, "fig6d_per_batch_f1", FIG_DIR)
        plt.show()
else:
    print("prediction_folds_long.csv is empty — panel skipped.")

In [ ]:
f3

### Additional violinplot

In [ ]:
f3[['batch', 'f1_macro']].groupby('batch').median().sort_values('f1_macro').index

In [ ]:
f3[(f3.batch.isin(['GPL8432_FFPE_Unknown', 'GPL20188_FF_Unknown', 'GPL17077_FF_Unknown',
       'GPL16686_FF_Unknown', 'GPL23541_FF_Unknown', 'GPL13158_FF_Unknown']))&(f3.is_best)][['batch', 'f1_macro']].groupby('batch').median()

In [ ]:
f3[(f3.batch.isin(['GPL14951_FFPE_Unknown', 'GPL570_FFPE_Unknown',
                   'GPL570_Unknown_Unknown', 'GPL570_FF_Unknown', 'RNASeq_FFPE_Exome_capture', 'RNASeq_FF_PolyA']))&(f3.is_best)][['batch', 'f1_macro']].groupby('batch').median()



In [ ]:
f3[(f3.batch.isin(['GPL14951_FFPE_Unknown', 'GPL570_FFPE_Unknown',
                   'GPL570_Unknown_Unknown', 'GPL570_FF_Unknown', 'RNASeq_FFPE_Exome_capture', 'RNASeq_FF_PolyA']))&(~f3.is_best)][['batch', 'f1_macro']].groupby('batch').median()


In [ ]:
for batch in ['GPL8432_FFPE_Unknown', 'GPL20188_FF_Unknown', 'GPL17077_FF_Unknown',
       'GPL16686_FF_Unknown', 'GPL23541_FF_Unknown', 'GPL13158_FF_Unknown']:
    best = f3[(f3.batch == batch)&(f3.is_best)]['f1_macro'].dropna()
    rest = f3[(f3.batch == batch)&(~f3.is_best)]['f1_macro'].dropna()
    
    u, p = mannwhitneyu(best, rest, alternative="two-sided")
    print(batch, p)

In [ ]:
for batch in ['GPL14951_FFPE_Unknown', 'GPL570_FFPE_Unknown',
                   'GPL570_Unknown_Unknown', 'GPL570_FF_Unknown', 'RNASeq_FFPE_Exome_capture', 'RNASeq_FF_PolyA']:
    best = f3[(f3.batch == batch)&(f3.is_best)]['f1_macro'].dropna()
    rest = f3[(f3.batch == batch)&(~f3.is_best)]['f1_macro'].dropna()
    
    u, p = mannwhitneyu(best, rest, alternative="two-sided")
    print(batch, p)

In [ ]:
f3.f1_macro.min()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
y_col = "f1_macro"
sns.violinplot(
    data=f3,#.dropna(),
    x="batch",
    y=y_col,
    hue="group",
    palette = {'Best (clustermap)': COL_BEST, 'Rest': COL_REST},
    split=True,
    gap=0,
    inner="quart",
    ax=ax,
    order=f3[['batch', y_col]].groupby('batch').median().sort_values(y_col).index,
    linewidth=0.025,
    width=1.3,
    #cut=0,
    bw_adjust=0.4,
    bw_method=0.4
)
plt.ylim(0, 1)
ax.tick_params(axis="x", rotation=90)
save_figure(fig, f"violin_plot_{y_col}_per_batch", FIG_DIR)

## §8 — Permutation negative control

Biological labels are shuffled globally and the whole leave-one-batch-out evaluation is
repeated. The expected result is a collapse to chance. If a permuted score stays well
above chance, that indicates the label collapse is confounded with batch — a batch that is
100 % DLBCL, for instance — and it is reported rather than tuned away.

The 260824 run used **20 permutations**, so the empirical p-value cannot fall below
`1/21 = 0.0476`. That floor sits just under the 0.05 line drawn below, and most attempts
will land exactly on it. The floor is a property of the permutation count, not evidence
of borderline significance; quote the observed-minus-permuted delta rather than the
p-value, or recompute with `--n-perm 100`.

In [ ]:
perm_pairs = [
    (c, c.replace("_mean", "_perm_mean"))
    for c in [
        "pv_lobo3_f1_macro_mean",
        "pv_lobo2_f1_macro_mean",
        "pv_lobo3_auc_macro_mean",
        "pv_lobo2_auc_macro_mean",
    ]
]
perm_pairs = [
    (o, p) for o, p in perm_pairs if o in df_lmn.columns and p in df_lmn.columns
]

if perm_pairs:
    d = df_lmn[~df_lmn["is_raw"]]
    fig, axes = plt.subplots(
        1, len(perm_pairs) + 1, figsize=(1.9 * (len(perm_pairs) + 1), 2.7)
    )
    axes = np.atleast_1d(axes)
    for ax, (obs, perm) in zip(axes, perm_pairs):
        sub = d[[obs, perm]].dropna()
        if sub.empty:
            ax.set_axis_off()
            continue
        ax.scatter(sub[perm], sub[obs], s=5, alpha=0.5, color=COL_REST, linewidths=0)
        best = d.loc[d["is_best"], [obs, perm]].dropna()
        ax.scatter(
            best[perm], best[obs], s=12, color=COL_BEST, linewidths=0, label="best"
        )
        lims = [min(sub.min()), max(sub.max())]
        ax.plot(lims, lims, ls="--", color="black", lw=0.8)
        ax.set_xlabel("permuted")
        ax.set_ylabel("observed")
        ax.set_title(obs.replace("pv_", "").replace("_mean", ""), loc="left")
        ax.legend(frameon=False, fontsize=FONT_SIZE - 1)

    pcols = [c for c in df_lmn.columns if c.endswith("_perm_pvalue")]
    if pcols:
        d_perm = d[d["pv_perm_comparable"]] if "pv_perm_comparable" in d else d
        long = d_perm[pcols].melt(var_name="metric", value_name="p").dropna()
        long["metric"] = long["metric"].str.replace("pv_", "", regex=False)
        sns.boxplot(
            data=long,
            x="metric",
            y="p",
            ax=axes[-1],
            showfliers=False,
            color=COL_REST,
            width=0.6,
        )
        axes[-1].axhline(0.05, color=COL_BEST, ls="--", lw=1, label="p = 0.05")
        axes[-1].axhline(
            1 / (PV_N_PERM_EXPECTED + 1),
            color=COL_RAW,
            ls=":",
            lw=1,
            label=f"floor 1/({PV_N_PERM_EXPECTED}+1)",
        )
        axes[-1].set_yscale("log")
        axes[-1].legend(frameon=False)
        axes[-1].tick_params(axis="x", labelrotation=30)
        axes[-1].set_xlabel("")
        axes[-1].set_title("Empirical p", loc="left")
    else:
        axes[-1].set_axis_off()

    fig.suptitle(
        "Permutation control: points below the diagonal would mean no signal", y=1.04
    )
    fig.tight_layout()
    save_figure(fig, "fig6e_permutation_control", FIG_DIR)
    plt.show()

    for obs, perm in perm_pairs:
        sub = d[[obs, perm]].dropna()
        if not sub.empty:
            print(
                f"{obs:32s} observed {sub[obs].median():.3f}  "
                f"permuted {sub[perm].median():.3f}  "
                f"delta {sub[obs].median() - sub[perm].median():+.3f}"
            )
else:
    print("Permutation columns not in the table yet — panel skipped.")

### Additional plots and statistics

In [ ]:
d

In [ ]:

median_abs_deviation(d["pv_lobo3_f1_macro_perm_mean"][~d['is_best']].dropna())

In [ ]:
d["pv_lobo3_f1_macro_perm_mean"][d['is_best']].median()

In [ ]:
d["pv_lobo3_f1_macro_perm_mean"][~d['is_best']].median()

In [ ]:
mannwhitneyu(d["pv_lobo3_f1_macro_perm_mean"][~d['is_best']].dropna(), 
            d["pv_lobo3_f1_macro_perm_mean"][d['is_best']].dropna())

In [ ]:
obs = "pv_lobo3_f1_macro_mean"
             
perm = "pv_lobo3_f1_macro_perm_mean"

fig, ax = plt.subplots(
    figsize=(2.7, 2.7)
    )

sub = d[['method', obs, perm]].dropna()

ax.scatter(sub[perm], sub[obs], s=5, alpha=0.25, color=COL_REST, linewidths=0)
best = d.loc[d["is_best"], ['method',obs, perm]].dropna()

for method in best.method.unique():
    best_current = best[best.method == method]
    ax.scatter(
                best_current[perm], best_current[obs], s=12, color=method_pal.get(method), linewidths=0, label="best"
            )
lims = [min(sub[[obs, perm]].min()), max(sub[[obs, perm]].max())]
ax.plot(lims, lims, ls="--", color="black", lw=0.8)
ax.set_xlabel("permuted")
ax.set_ylabel("observed")
ax.set_title(obs.replace("pv_", "").replace("_mean", ""), loc="left")
ax.legend(frameon=False, fontsize=FONT_SIZE - 1)

fig.tight_layout()
save_figure(fig, f"fig6e_permutation_control_{obs}", FIG_DIR)
plt.show()

## §9 — Supplementary File 5 export

Exports the wide blind-check table, the three long-format detail tables, the gene
annotation and the panel summary. Figures are already saved as both PDF and SVG by
`save_figure()`.

In [ ]:
SUPP_DIR = FIG_DIR / f"supplementary_file_5{MK_SFX}"
SUPP_DIR.mkdir(parents=True, exist_ok=True)

# Per-gene_group summary for the supplementary table — one row per signature.
panel_summary_table = (
    marker_ann.groupby("gene_group")
    .agg(
        n_genes=("gene", "nunique"),
        cell_type=("cell_type", lambda s: sorted(set(s))[0]),
        pathway=("pathway", lambda s: sorted(set(s))[0]),
        tme_subtype=("tme_subtype", lambda s: sorted(set(s))[0]),
        prognostic_significance=(
            "prognostic_significance",
            lambda s: sorted(set(s))[0],
        ),
        source_article=("source_article", lambda s: sorted(set(s))[0]),
    )
    .reset_index()
    .sort_values("gene_group")
)

exports = {
    "S5_1_blind_check_metrics_wide.csv": df_lmn.reset_index(),
    "S5_2_marker_gene_correlations_long.csv": gene_long,
    "S5_3_marker_cohort_correlations_long.csv": cohort_long,
    "S5_4_prediction_folds_long.csv": folds_long,
    "S5_5_marker_gene_annotation.csv": marker_ann,
    "S5_6_marker_panel_summary.csv": panel_summary_table,
}
for name, table in exports.items():
    if table is None or len(table) == 0:
        print(f"  SKIP {name} (empty)")
        continue
    table.to_csv(SUPP_DIR / name, index=False)
    print(f"  wrote {name}  ({len(table)} rows)")

if COMPLETE_GENES:
    pd.DataFrame({"gene": COMPLETE_GENES}).to_csv(
        SUPP_DIR / "S5_7_genes_complete_across_attempts.csv", index=False
    )
    print(
        f"  wrote S5_7_genes_complete_across_attempts.csv ({len(COMPLETE_GENES)} genes)"
    )

if NARROW:
    marker_ann.loc[
        marker_ann["in_narrow_set"],
        ["gene", "gene_group", "cell_type", "is_housekeeping"],
    ].to_csv(SUPP_DIR / "S5_8_narrow_panel_genes.csv", index=False)
    print(f"  wrote S5_8_narrow_panel_genes.csv ({len(NARROW_GENES)} genes)")
    if "noise_cmp" in globals():
        noise_cmp.to_csv(SUPP_DIR / "S5_9_hk_control_noise_comparison.csv", index=False)
        print("  wrote S5_9_hk_control_noise_comparison.csv")

print(f"\nSupplementary File 5 staged in {SUPP_DIR}")

---

### Numbers to carry into the manuscript

Every value quoted in the Results must come from a named panel above or from
Supplementary File 5. In particular the manuscript must state:

1. Group L saturates at ~1.0 for per-batch monotone methods — the ceiling is a property of
   the metric, so a high value is not evidence of quality (§2, §3).
2. Marker coverage per gene, including that CD20, CD3, CD10, CD23, CD79A/B and PD-1 are
   absent from the strict intersection but present in about two-thirds of `(strat, imp)`
   pairs (§5).
3. The Group M biology margin, not the raw agreement (§6, panels B and C).
4. `n_folds` and `n_folds_multiclass` alongside every predictive mean (§7).
5. Whether the permutation control collapsed to chance (§8).

Manuscript writing is deliberately deferred until these results exist — see
`project_manuscript_lmn_sections_deferred.md` in project memory.